#Silver Layer — Cleaning + SCD Type 2

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("healthcare_catalog.bronze.patients")

df_clean = (
    df_bronze
    # 1. Remove exact duplicate business records (ignore lineage columns)
    .dropDuplicates(["Name", "Age", "Gender", "Date_of_Admission", "Hospital", "Doctor"])

    # 2. Drop rows missing critical fields
    .filter(
        F.col("Name").isNotNull() &
        F.col("Date_of_Admission").isNotNull() &
        F.col("Hospital").isNotNull()
    )

    # 3. Standardize text casing
    .withColumn("Name", F.initcap(F.trim(F.col("Name"))))
    .withColumn("Hospital", F.initcap(F.trim(F.col("Hospital"))))
    .withColumn("Doctor", F.initcap(F.trim(F.col("Doctor"))))
    .withColumn("Insurance_Provider", F.initcap(F.trim(F.col("Insurance_Provider"))))
    .withColumn("Medical_Condition", F.initcap(F.trim(F.col("Medical_Condition"))))
    .withColumn("Admission_Type", F.initcap(F.trim(F.col("Admission_Type"))))
    .withColumn("Medication", F.initcap(F.trim(F.col("Medication"))))
    .withColumn("Test_Results", F.initcap(F.trim(F.col("Test_Results"))))
    .withColumn("Gender", F.initcap(F.trim(F.col("Gender"))))
    .withColumn("Blood_Type", F.upper(F.trim(F.col("Blood_Type"))))

    # 4. Fix data types
    .withColumn("Age", F.col("Age").cast("int"))
    .withColumn("Billing_Amount", F.round(F.col("Billing_Amount").cast("double"), 2))
    .withColumn("Room_Number", F.col("Room_Number").cast("int"))

    # 5. Standardize dates to proper date type
    .withColumn("Date_of_Admission", F.to_date("Date_of_Admission", "yyyy-MM-dd"))
    .withColumn("Discharge_Date", F.to_date("Discharge_Date", "yyyy-MM-dd"))

    # 6. Drop rows where type coercion failed, or values are implausible
    .filter(
        F.col("Age").isNotNull() &
        F.col("Date_of_Admission").isNotNull() &
        F.col("Billing_Amount").isNotNull() &
        (F.col("Age") >= 0) & (F.col("Age") <= 120) &
        (F.col("Billing_Amount") >= 0)
    )
)

print(f"Bronze rows: {df_bronze.count():,}")
print(f"Clean rows after Silver rules: {df_clean.count():,}")
display(df_clean.limit(5))

#SCD Type 2 via MERGE INTO

In [0]:
from delta.tables import DeltaTable

silver_table_name = "healthcare_catalog.silver.patients"

# First run: Silver table doesn't exist yet, so we SEED it —
# every row starts as "version 1, currently active"
table_exists = spark.catalog.tableExists(silver_table_name)

if not table_exists:
    spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_catalog.silver")

    df_seed = (
        df_clean
        .withColumn("scd_effective_start_date", F.col("Date_of_Admission"))
        .withColumn("scd_effective_end_date", F.lit(None).cast("date"))
        .withColumn("scd_is_current", F.lit("Y"))
        .withColumn("_silver_processed_timestamp", F.current_timestamp())
    )

    df_seed.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    print(f"Silver table seeded with {df_seed.count():,} rows (first load)")

else:
    # Subsequent runs: MERGE — close out changed rows, insert new ones
    silver_table = DeltaTable.forName(spark, silver_table_name)

    # Business key: a patient's identity for this pipeline =
    # Name + admission date (the natural key of "one hospital visit")
    merge_condition = """
        target.Name = source.Name
        AND target.Date_of_Admission = source.Date_of_Admission
        AND target.scd_is_current = 'Y'
    """

    (
        silver_table.alias("target")
        .merge(df_clean.alias("source"), merge_condition)
        .whenMatchedUpdate(
            condition="""
                target.Discharge_Date IS DISTINCT FROM source.Discharge_Date
                OR target.Test_Results IS DISTINCT FROM source.Test_Results
                OR target.Billing_Amount IS DISTINCT FROM source.Billing_Amount
            """,
            set={
                "scd_effective_end_date": "current_date()",
                "scd_is_current": "'N'"
            }
        )
        .whenNotMatchedInsert(values={
            "Name": "source.Name",
            "Age": "source.Age",
            "Gender": "source.Gender",
            "Blood_Type": "source.Blood_Type",
            "Medical_Condition": "source.Medical_Condition",
            "Date_of_Admission": "source.Date_of_Admission",
            "Doctor": "source.Doctor",
            "Hospital": "source.Hospital",
            "Insurance_Provider": "source.Insurance_Provider",
            "Billing_Amount": "source.Billing_Amount",
            "Room_Number": "source.Room_Number",
            "Admission_Type": "source.Admission_Type",
            "Discharge_Date": "source.Discharge_Date",
            "Medication": "source.Medication",
            "Test_Results": "source.Test_Results",
            "_ingestion_timestamp": "source._ingestion_timestamp",
            "_source_file": "source._source_file",
            "scd_effective_start_date": "source.Date_of_Admission",
            "scd_effective_end_date": "CAST(NULL AS DATE)",
            "scd_is_current": "'Y'"
        })
        .execute()
    )
    print("MERGE complete — Silver table updated with SCD Type 2 logic")

In [0]:
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {silver_table_name}"))
display(spark.sql(f"SELECT scd_is_current, COUNT(*) FROM {silver_table_name} GROUP BY scd_is_current"))